# FinIA-Flex — Paso 4: Prompt Maestro

Caso Práctico Unidad 1, materia Generative IA, IEP.

Aquí diseño y pruebo el prompt maestro solo, sin conectar todavía el RAG (eso es el Paso 5) —
le paso el contexto de política a mano, para saber si algo sale mal, si es culpa del prompt o
de la búsqueda.

Técnicas que uso:
1. Role prompting — le doy el rol de analista financiero senior.
2. Estructura de salida obligatoria — 5 secciones fijas.
3. Few-shot — un ejemplo resuelto para que copie el formato.
4. Chain-of-thought — le pido razonar internamente antes de redactar.
5. Grounding — que solo cite política si de verdad está en el contexto, y que diga si no hay
   nada aplicable en vez de inventar.

Uso la API gratuita de Groq (Llama) para no gastar nada en esta etapa. El prompt no depende
del proveedor — funcionaría igual con Claude u OpenAI cambiando solo la celda de conexión.


## 1. Instalación y configuración

Se requiere una API key gratuita de Groq (https://console.groq.com/keys). El valor no debe
escribirse directamente en el notebook; se solicita de forma segura con `getpass` para evitar
que quede expuesto en el archivo o en el repositorio de evidencias.


In [ ]:
!pip install -q groq

from getpass import getpass
import os

os.environ["GROQ_API_KEY"] = getpass("Ingresar API key de Groq: ")

## 2. El prompt maestro

Este es el prompt que arma el rol, la estructura y las reglas de razonamiento. Está basado en
mis propios documentos POL-FIN-002 y POL-FIN-003 del Paso 2 — no es genérico, refleja los
criterios reales que definí ahí.


In [ ]:
SYSTEM_PROMPT = """Eres un analista financiero senior de FlexParts Manufacturing MX,
especializado en control de costos de manufactura. Tu tarea es generar reportes ejecutivos
de variación presupuestal para Gerencia, a partir de datos de presupuesto vs. gasto real y
del contexto de políticas internas que se te proporcione.

RAZONAMIENTO INTERNO (no lo muestres en la respuesta final, solo úsalo para pensar):
1. Clasifica cada variación según su magnitud: dentro de rango normal (-2.9% a +2.9%),
   moderada (+3% a +9.9%), significativa (+10% o más), ahorro saludable (-10% a -3%), o
   ahorro atípico (menor a -10%).
2. Verifica si la variación, por su magnitud o por ser sostenida 3 meses o más, activa
   alguna regla del contexto de políticas proporcionado. Si el contexto no incluye una
   política aplicable, indícalo explícitamente — nunca inventes un umbral o una regla que no
   esté en el contexto.
3. Distingue causas internas (atendibles por el responsable del centro de costo) de causas
   externas (fuera de su control), cuando el contexto lo permita.
4. Solo después de este análisis, redacta el reporte final.

ESTRUCTURA OBLIGATORIA DE LA RESPUESTA FINAL:
1. Resumen Ejecutivo (máximo 3 líneas)
2. Diagnóstico por Centro de Costo (variación en MXN y %, clasificación, causa probable)
3. Alertas de Política (solo si el contexto proporcionado activa alguna; si no, escribir
   "Sin alertas de política en el contexto disponible")
4. Recomendación (una acción concreta y accionable por cada hallazgo relevante)
5. Responsable y Siguiente Paso

REGLAS DE GROUNDING:
- Usa exclusivamente los datos numéricos y el contexto de políticas que se te proporcionen.
- Si citas una política o un umbral, debe provenir textualmente del contexto recibido.
- Si el contexto no cubre algo que sería útil mencionar, indica la limitación en vez de
  completar con supuestos.
- NUNCA inventes nombres de personas, cargos o responsables. El campo "Responsable" debe
  llenarse únicamente con el valor recibido en los DATOS de entrada, copiado tal cual. Si el
  campo Responsable no viene incluido en los DATOS, escribe exactamente "No especificado en
  los datos proporcionados" — no propongas un nombre, cargo o departamento por tu cuenta bajo
  ninguna circunstancia.

TONO: profesional, directo, sin tecnicismos innecesarios. El reporte debe ser legible para
un Gerente de Planta que no es especialista financiero. Evita juicios de valor sobre las
personas; evalúa procesos y resultados.
"""

print(SYSTEM_PROMPT)

## 3. Ejemplo few-shot

Tomo un ejemplo de mi documento POL-FIN-003 para que el modelo tenga algo concreto que
imitar antes de meterle el caso real.


In [ ]:
EJEMPLO_FEW_SHOT_ENTRADA = """
DATOS:
Centro de costo: Línea de Producción 2
Categoría: Mantenimiento
Presupuesto: $70,000 MXN | Real: $87,200 MXN (Septiembre)
Histórico: Julio +18.6%, Agosto +20.0%, Septiembre +24.6%
Responsable: Coordinador de Mantenimiento - J. Salinas

CONTEXTO DE POLÍTICAS RECUPERADO:
"Cuando una categoría de gasto en un mismo centro de costo presenta una variación positiva
(sobrecosto) durante 3 meses consecutivos o más, el responsable debe presentar un plan
correctivo formal a Gerencia, independientemente de si cada mes individual superó o no el
umbral de aprobación." (Política POL-FIN-001, sección 4)
"""

EJEMPLO_FEW_SHOT_SALIDA = """
1. Resumen Ejecutivo
Mantenimiento en Línea de Producción 2 muestra sobrecosto sostenido por tercer mes
consecutivo, activando la regla de variación sostenida de la Política POL-FIN-001.

2. Diagnóstico por Centro de Costo
- Línea de Producción 2 / Mantenimiento: variación de +$17,200 MXN (+24.6%) en septiembre.
  Clasificación: significativa. Tendencia sostenida desde julio (+18.6%, +20.0%, +24.6%).

3. Alertas de Política
Se activa la regla de variación sostenida (POL-FIN-001, sección 4): 3 meses consecutivos de
sobrecosto en la misma categoría y centro de costo requieren plan correctivo formal a
Gerencia, independientemente del monto individual de cada mes.

4. Recomendación
Solicitar al Coordinador de Mantenimiento un plan correctivo formal antes del cierre del
siguiente mes, desagregando el gasto entre mantenimiento correctivo y preventivo para
identificar si el sobrecosto responde a fallas puntuales o a un patrón estructural.

5. Responsable y Siguiente Paso
Responsable: Coordinador de Mantenimiento - J. Salinas.
Siguiente paso: presentar plan correctivo formal a Gerencia — fecha límite sugerida: cierre
del mes en curso.
"""

print("Ejemplo few-shot definido.")

## 4. Probando con un caso real (sin RAG todavía)

Pruebo con el escenario de ahorro atípico (Línea 1 / Materia Prima), pegando el contexto a
mano — en el Paso 5 eso ya lo va a hacer solo.

Aquí me pasó algo interesante: en una prueba anterior no incluí el campo Responsable en los
datos, y el modelo se inventó un nombre que no existía en ningún lado. Revisando, me di
cuenta de que el propio ejemplo few-shot tampoco traía ese campo en la entrada — el modelo
aprendió, por imitación, que estaba bien inventarlo. Lo arreglé agregando Responsable al
few-shot y una regla explícita en el prompt que prohíbe inventar nombres.


In [ ]:
CASO_PRUEBA_ENTRADA = """
DATOS:
Centro de costo: Línea de Producción 1
Categoría: Materia Prima
Presupuesto: $620,000 MXN | Real: $565,100 MXN (Enero)
Histórico: variación entre -7% y -9% durante todo el año.
Responsable: Gerente de Línea 1 - R. Hernández

CONTEXTO DE POLÍTICAS RECUPERADO:
"Un ahorro sostenido y significativo (por debajo de -10%) en categorías como Mantenimiento
puede ser señal de mantenimiento diferido, lo cual representa un riesgo operativo futuro
aunque mejore el resultado financiero del mes. Todo análisis de ahorro debe evaluarse junto
con indicadores operativos, no solo financieros." (Política POL-FIN-002, sección 4)
"""

print(CASO_PRUEBA_ENTRADA)

In [ ]:
from groq import Groq

client = Groq()

respuesta = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": EJEMPLO_FEW_SHOT_ENTRADA},
        {"role": "assistant", "content": EJEMPLO_FEW_SHOT_SALIDA},
        {"role": "user", "content": CASO_PRUEBA_ENTRADA},
    ],
    temperature=0.3,
)

print(respuesta.choices[0].message.content)

## 5. Revisando el resultado

Cosas que reviso a mano en la respuesta de arriba (esto es la base de lo que después
automatizo en el Paso 7):

- ¿Tiene las 5 secciones?
- ¿La variación en MXN y % coincide con los datos que le di?
- ¿La política que cita de verdad está en el contexto, o se la inventó? (aquí no debería
  mencionar el umbral de Mantenimiento, porque este caso es de Materia Prima)
- ¿El Responsable es exactamente el que le di, sin inventar nada?
- ¿La recomendación es algo concreto, o solo repite el diagnóstico?


---
## Resumen (Paso 4)

**Qué hice:** diseñé un prompt maestro con role prompting, estructura de salida
obligatoria, few-shot prompting y chain-of-thought guiado, probado de forma aislada (sin RAG
automático) usando la API gratuita de Groq (modelo Llama 3.3 70B).

Le pido razonar internamente pero sin mostrarlo en la respuesta final, para que el reporte
quede limpio. Las reglas de grounding (no inventar políticas ni responsables) ya las meto
desde aquí, aunque el control de calidad automatizado lo formalizo hasta el Paso 7.

El hallazgo del responsable inventado (arriba, Sección 4) ya quedó corregido — la causa fue
que mi propio few-shot no traía ese dato en la entrada, y lo arreglé.

Siguiente: Paso 5, conectar este prompt con el índice del Paso 3 para que la búsqueda de
contexto ya no sea manual.
